# Jaipur Junction-Informed Railway Platform Allocation: Three-Algorithm Benchmark

**Research comparison:** Exact Classical Search vs Classical Simulated Annealing (SA) vs Simulated Quantum Annealing (SQA).

This notebook creates a *historically calibrated synthetic benchmark*, not a reconstruction of Jaipur Junction's current working timetable or yard. Public historical inputs used to set the benchmark profile are: six candidate platform resources; 28 trains in the historical busiest six-hour period; and a sensitivity case with four or five long-train-capable platform resources. The train-level arrival/departure times, nominal assignments and disruptions are generated synthetically.

## Five model elements
1. Exactly one eligible platform per train.
2. No same-platform overlap after an assumed clearance buffer.
3. 24+ coach trains restricted to a chosen set of 4 or 5 long-train-capable resources.
4. Minimize the number of changes from the nominal platform assignment.
5. Explicitly generate delayed arrival/departure occupation intervals.

**Important modelling assumptions:** the identity of the four/five long-train-capable resources is not known from the historical aggregate data; this notebook uses abstract platform IDs `P0...P5` and assumes the first 4 or 5 are long-train-capable. The 5-minute base clearance and delay levels are experimental settings, not claimed Jaipur operating rules. Actual Jaipur SWR, yard diagram and interlocking data are not available in this benchmark.

Exact search enumerates legal one-hot assignments using backtracking and conflict pruning. It is only run on small instances. SA and SQA use the same QUBO and same instances. OpenJij samplers execute on CPU; the presence of a GPU runtime does not make these calls GPU-accelerated.

In [4]:
%pip install openjij pyqubo pandas numpy matplotlib scipy psutil

  Using cached dimod-0.12.23-cp312-abi3-win_amd64.whl.metadata (4.1 kB)
  Using cached dwave_neal-0.6.0-py3-none-any.whl.metadata (3.0 kB)
  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached dwave_samplers-1.8.0-cp312-abi3-win_amd64.whl.metadata (24 kB)
   ---------------------------------------- 0.0/849.5 kB ? eta -:--:--
   ---------------------------------------- 849.5/849.5 kB 7.4 MB/s  0:00:00
Using cached dimod-0.12.23-cp312-abi3-win_amd64.whl (2.4 MB)
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   -------------------------------------- - 1.0/1.1 MB 10.1 MB/s eta 0:00:01
   ---------------------------------------- 1.1/1.1 MB 2.2 MB/s  0:00:00
   ---------------------------------------- 0.0/36.7 MB ? eta -:--:--
   -- ------------------------------------- 2.1/36.7 MB 13.1 MB/s eta 0:00:03
   ----- ---------------------------------- 4.7/36.7 MB 11.4 MB/s eta 0:00:03


In [5]:
# If running in a fresh Colab runtime, uncomment this line:
# %pip -q install openjij numpy pandas matplotlib

import sys, platform, time, itertools, math, json, os
from dataclasses import dataclass
from typing import List, Dict, Tuple
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import openjij as oj
    print("OpenJij:", oj.__version__ if hasattr(oj, "__version__") else "version not exposed")
except ImportError as e:
    raise ImportError("Install OpenJij first: %pip install openjij") from e

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("CPU:", platform.processor() or "not exposed")
try:
    import psutil
    print("RAM (GB):", round(psutil.virtual_memory().total/1024**3, 2))
except Exception:
    print("RAM: not measured (psutil unavailable)")
try:
    import torch
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))
except Exception:
    print("PyTorch/CUDA: not installed")
print("NOTE: OpenJij SA/SQA below are CPU sampler calls in this setup.")

OpenJij: version not exposed
Python: 3.12.12
Platform: Windows-11-10.0.26300-SP0
CPU: Intel64 Family 6 Model 143 Stepping 8, GenuineIntel
RAM (GB): 127.65
PyTorch/CUDA: not installed
NOTE: OpenJij SA/SQA below are CPU sampler calls in this setup.


## 1. Benchmark configuration

Start with a small smoke test. Increase the sizes only after validating the generated instances and solver outputs. The full 28-train run can be expensive, especially for SQA.

In [6]:
# Historical-profile benchmark settings (train-level records remain synthetic)
N_PLATFORMS = 6
PEAK_WINDOW_START = 12 * 60       # minutes after midnight
PEAK_WINDOW_END = 18 * 60
BASE_CLEARANCE_MIN = 5            # modelling assumption; test sensitivity
LONG_TRAIN_COACH_THRESHOLD = 24
CENTRAL_LONG_TRAIN_COUNT = 5      # proportional estimate, not observed peak-window count
LONG_PLATFORM_COUNTS = (4, 5)     # sensitivity scenarios

# Pilot defaults. Expand after a successful pilot.
TRAIN_COUNTS = [4, 6, 8, 10, 12, 16, 20, 24, 28]
DELAY_LEVELS_MIN = [0, 5, 10, 15]
DATASET_SEEDS = [11, 22, 33]       # pilot; use 10+ seeds for final paper
SOLVER_REPEATS = 3                # now actually used below; use 10+ for final paper
NUM_READS = 100                   # pilot; final target 300
NUM_SWEEPS = 500                  # pilot; final target 1000
EXACT_TIME_LIMIT_S = 60            # pilot; final target may be 300 seconds
PENALTY_LAMBDA = 20.0              # pilot value only; tune using penalty experiment

RUN_FULL_PILOT = True  # Executes repeated small-instance pilot only; no 28-train run
RUN_SQA = True


## 2. Generate a nominally feasible schedule, then inject independent delays

The nominal platform assignment is created first, with non-overlapping planned occupation on each assigned platform. Each train then receives an independently sampled delay from the selected controlled delay level. Arrival and departure are shifted together, preserving dwell time. If disruptions make the instance infeasible, it is retained and reported as infeasible; it is not silently discarded.

In [7]:
def generate_jaipur_profile_instance(
    n_trains: int,
    seed: int,
    long_platform_count: int = 4,
    long_train_count: int | None = None,
    delay_max_min: int = 10,
    clearance_min: int = BASE_CLEARANCE_MIN,
    n_platforms: int = N_PLATFORMS,
):
    """Generate a reproducible synthetic instance calibrated to Jaipur historical profile.

    Abstract assumption: P0..P(long_platform_count-1) are long-train-capable.
    Short trains are assumed eligible for all six resources.
    """
    if not 1 <= long_platform_count <= n_platforms:
        raise ValueError("long_platform_count must be between 1 and n_platforms")
    rng = np.random.default_rng(seed)
    if long_train_count is None:
        long_train_count = min(n_trains, max(1, round(n_trains * 18 / 97)))
    long_train_count = min(long_train_count, n_trains)
    long_flags = np.array([True]*long_train_count + [False]*(n_trains-long_train_count))
    rng.shuffle(long_flags)

    long_eligible = set(range(long_platform_count))
    eligible = [sorted(long_eligible) if is_long else list(range(n_platforms)) for is_long in long_flags]

    # Assign a nominal eligible platform while balancing train counts.
    nominal_platforms = []
    platform_load = np.zeros(n_platforms, dtype=int)
    for i in range(n_trains):
        candidates = eligible[i]
        min_load = min(platform_load[p] for p in candidates)
        best = [p for p in candidates if platform_load[p] == min_load]
        p = int(rng.choice(best))
        nominal_platforms.append(p)
        platform_load[p] += 1

    # Construct non-overlapping nominal occupation intervals on each nominal platform.
    # Spread each platform's trains across the six-hour window.
    arr_nom = np.zeros(n_trains, dtype=int)
    dep_nom = np.zeros(n_trains, dtype=int)
    for p in range(n_platforms):
        idxs = [i for i, pp in enumerate(nominal_platforms) if pp == p]
        if not idxs: continue
        # Sequential slots guarantee nominal feasibility including clearance.
        cursor = PEAK_WINDOW_START + int(rng.integers(0, 20))
        for i in idxs:
            dwell = int(rng.integers(10, 26))
            # Small random gaps, with no nominal overlap.
            cursor += int(rng.integers(2, 9))
            arr_nom[i] = cursor
            dep_nom[i] = cursor + dwell
            cursor = dep_nom[i] + clearance_min
        # If a platform's sequence extends beyond the window, rescale its slot spacing.
        if idxs and dep_nom[idxs[-1]] > PEAK_WINDOW_END:
            total_span = dep_nom[idxs[-1]] - arr_nom[idxs[0]]
            available = PEAK_WINDOW_END - PEAK_WINDOW_START - 30
            factor = available / max(total_span, 1)
            # Rebuild using compact dwell/gap values rather than clipping intervals.
            cursor = PEAK_WINDOW_START + 5
            for i in idxs:
                dwell = int(rng.integers(8, 17))
                arr_nom[i] = cursor
                dep_nom[i] = cursor + dwell
                cursor = dep_nom[i] + clearance_min + 1

    # Independently inject delays. The delayed times may extend beyond 18:00.
    delays = rng.integers(0, delay_max_min + 1, size=n_trains)
    arr_actual = arr_nom + delays
    dep_actual = dep_nom + delays

    rows = []
    for i in range(n_trains):
        rows.append({
            "train_id": f"J{i+1:02d}",
            "coaches": int(rng.choice([12, 16, 18, 22, 24, 26], p=[.08,.15,.22,.20,.22,.13])),
            "is_long": bool(long_flags[i]),
            "nominal_platform": int(nominal_platforms[i]),
            "eligible_platforms": eligible[i],
            "scheduled_arrival_min": int(arr_nom[i]),
            "scheduled_departure_min": int(dep_nom[i]),
            "delay_min": int(delays[i]),
            "actual_arrival_min": int(arr_actual[i]),
            "actual_departure_min": int(dep_actual[i]),
        })
    df = pd.DataFrame(rows)
    # Ensure coach count and long flag are consistent for this synthetic scenario.
    df["coaches"] = np.where(df["is_long"], np.maximum(df["coaches"], 24), np.minimum(df["coaches"], 22))
    return df


def find_conflicts(df: pd.DataFrame, clearance_min: int):
    conflicts = []
    for i in range(len(df)):
        si = int(df.iloc[i].actual_arrival_min)
        ei = int(df.iloc[i].actual_departure_min) + clearance_min
        for j in range(i+1, len(df)):
            sj = int(df.iloc[j].actual_arrival_min)
            ej = int(df.iloc[j].actual_departure_min) + clearance_min
            # Half-open intervals: touching endpoints after clearance are non-conflicting.
            if max(si, sj) < min(ei, ej):
                conflicts.append((i, j))
    return conflicts

# Preview one Jaipur-profile instance
preview = generate_jaipur_profile_instance(8, seed=11, long_platform_count=4, long_train_count=2, delay_max_min=10)
preview[["train_id","coaches","nominal_platform","eligible_platforms","scheduled_arrival_min","scheduled_departure_min","delay_min","actual_arrival_min","actual_departure_min"]]

,train_id,coaches,nominal_platform,eligible_platforms,scheduled_arrival_min,scheduled_departure_min,delay_min,actual_arrival_min,actual_departure_min
0,J01,22,0,"[0, 1, 2, 3, 4, 5]",743,768,9,752,777
1,J02,24,2,"[0, 1, 2, 3]",736,756,6,742,762
2,J03,22,5,"[0, 1, 2, 3, 4, 5]",737,762,10,747,772
3,J04,18,3,"[0, 1, 2, 3, 4, 5]",729,744,10,739,754
4,J05,22,1,"[0, 1, 2, 3, 4, 5]",732,744,1,733,745
5,J06,22,4,"[0, 1, 2, 3, 4, 5]",729,749,2,731,751
6,J07,24,2,"[0, 1, 2, 3]",768,782,3,771,785
7,J08,22,0,"[0, 1, 2, 3, 4, 5]",781,800,6,787,806


## 3. Independent feasibility validator

The validator checks the original constraints directly. A low QUBO energy alone is not accepted as proof of a valid allocation.

In [8]:
def validate_assignment(assignment: Dict[int, int], df: pd.DataFrame, conflicts, clearance_min: int):
    n = len(df)
    if set(assignment) != set(range(n)):
        return False, ["not_every_train_assigned"]
    violations = []
    for i, p in assignment.items():
        if p not in df.iloc[i].eligible_platforms:
            violations.append(f"ineligible_platform_train_{i}")
    for i, j in conflicts:
        if assignment.get(i) == assignment.get(j):
            violations.append(f"platform_conflict_{i}_{j}")
    return len(violations) == 0, violations


def nominal_assignment(df):
    return {i: int(df.iloc[i].nominal_platform) for i in range(len(df))}

# Verify nominal schedule is conflict-free before disruptions (using scheduled times).
def scheduled_conflicts(df, clearance_min):
    conflicts=[]
    for i in range(len(df)):
        si=int(df.iloc[i].scheduled_arrival_min); ei=int(df.iloc[i].scheduled_departure_min)+clearance_min
        for j in range(i+1,len(df)):
            sj=int(df.iloc[j].scheduled_arrival_min); ej=int(df.iloc[j].scheduled_departure_min)+clearance_min
            if max(si,sj)<min(ei,ej) and int(df.iloc[i].nominal_platform)==int(df.iloc[j].nominal_platform):
                conflicts.append((i,j))
    return conflicts

nominal_conflicts = scheduled_conflicts(preview, BASE_CLEARANCE_MIN)
print("Nominal same-platform conflicts:", nominal_conflicts)
print("Delayed conflict pairs:", find_conflicts(preview, BASE_CLEARANCE_MIN))

Nominal same-platform conflicts: []
Delayed conflict pairs: [(0, 1), (0, 2), (0, 3), (0, 5), (0, 6), (1, 2), (1, 3), (1, 4), (1, 5), (2, 3), (2, 4), (2, 5), (2, 6), (3, 4), (3, 5), (4, 5), (6, 7)]


## 4. Build the reduced-variable QUBO

Only eligible train-platform variables are created. The objective is the number of trains moved from their nominal platform: cost 0 for retaining it, cost 1 for changing it. Assignment and same-platform conflict constraints are encoded as quadratic penalties.

In [9]:
def build_qubo(df: pd.DataFrame, conflicts, penalty: float):
    labels = [(i, int(p)) for i in range(len(df)) for p in df.iloc[i].eligible_platforms]
    idx = {label:k for k,label in enumerate(labels)}
    Q = {}
    offset = 0.0
    def add(a,b,value):
        key = (a,b) if a <= b else (b,a)
        Q[key] = Q.get(key, 0.0) + float(value)

    # Objective: count platform changes.
    for i,p in labels:
        add(idx[(i,p)], idx[(i,p)], 0.0 if p == int(df.iloc[i].nominal_platform) else 1.0)

    # One-hot penalty: lambda*(sum_p x_ip - 1)^2
    # Binary simplification: lambda*(1 - sum x + 2*sum_{p<q} x_p*x_q)
    offset += penalty * len(df)
    for i in range(len(df)):
        vars_i = [idx[(i,int(p))] for p in df.iloc[i].eligible_platforms]
        for a in vars_i: add(a,a,-penalty)
        for a,b in itertools.combinations(vars_i,2): add(a,b,2*penalty)

    # Pairwise conflict penalty only on common eligible platforms.
    for i,j in conflicts:
        common = sorted(set(df.iloc[i].eligible_platforms).intersection(df.iloc[j].eligible_platforms))
        for p in common:
            add(idx[(i,p)], idx[(j,p)], penalty)

    # OpenJij QUBO dictionary uses variable labels. Keep only nonzero terms.
    Q_named = {}
    for (a,b),v in Q.items():
        if abs(v) > 1e-12:
            Q_named[(f'x_{labels[a][0]}_{labels[a][1]}', f'x_{labels[b][0]}_{labels[b][1]}')] = v
    return Q_named, labels, offset

qubo, variable_labels, qubo_offset = build_qubo(preview, find_conflicts(preview, BASE_CLEARANCE_MIN), PENALTY_LAMBDA)
print("Eligible binary variables:", len(variable_labels))
print("Nonzero QUBO terms:", len(qubo))
print("QUBO offset:", qubo_offset)

Eligible binary variables: 44
Nonzero QUBO terms: 232
QUBO offset: 160.0


In [10]:
def direct_model_energy(bit_values, df, conflicts, penalty):
    """Evaluate objective + penalties directly, independently of QUBO coefficients."""
    assignment_bits = {}
    for i in range(len(df)):
        assignment_bits[i] = [int(bit_values.get((i, int(p)), 0)) for p in df.iloc[i].eligible_platforms]
    objective = sum(
        int(bit_values.get((i, int(p)), 0))
        for i in range(len(df)) for p in df.iloc[i].eligible_platforms
        if int(p) != int(df.iloc[i].nominal_platform)
    )
    one_hot = sum((sum(bits) - 1) ** 2 for bits in assignment_bits.values())
    conflict_penalty = 0
    for i, j in conflicts:
        common = set(df.iloc[i].eligible_platforms).intersection(df.iloc[j].eligible_platforms)
        conflict_penalty += sum(int(bit_values.get((i, int(p)), 0)) * int(bit_values.get((j, int(p)), 0)) for p in common)
    return float(objective + penalty * (one_hot + conflict_penalty))


def qubo_energy(bit_values, Q_named, labels, offset):
    x = {f'x_{i}_{p}': int(bit_values.get((i, p), 0)) for i, p in labels}
    energy = float(offset)
    for (u, v), coefficient in Q_named.items():
        energy += float(coefficient) * x[u] * x[v]
    return energy


def verify_qubo_identity(df, conflicts, penalty=17.0, trials=250, seed=2026, atol=1e-8):
    """Random-bitstring test: QUBO energy must equal direct model energy, even if infeasible."""
    Q, labels, offset = build_qubo(df, conflicts, penalty)
    rng = np.random.default_rng(seed)
    max_abs_error = 0.0
    for _ in range(trials):
        bits = {(i, int(p)): int(rng.integers(0, 2)) for i in range(len(df)) for p in df.iloc[i].eligible_platforms}
        e_direct = direct_model_energy(bits, df, conflicts, penalty)
        e_qubo = qubo_energy(bits, Q, labels, offset)
        max_abs_error = max(max_abs_error, abs(e_direct - e_qubo))
        if not np.isclose(e_direct, e_qubo, atol=atol, rtol=0):
            raise AssertionError(f'QUBO mismatch: direct={e_direct}, QUBO={e_qubo}')
    return {"passed": True, "trials": trials, "max_abs_error": max_abs_error, "variables": len(labels)}

qubo_check = verify_qubo_identity(preview, find_conflicts(preview, BASE_CLEARANCE_MIN))
print('QUBO identity check:', qubo_check)


QUBO identity check: {'passed': True, 'trials': 250, 'max_abs_error': 0.0, 'variables': 44}


## 5. Exact Classical Search with conflict pruning

This is exact backtracking over legal platform choices. It stops at a user-defined time limit. A completed search proves the optimum; a timeout does not.

In [11]:
def exact_backtracking(df, conflicts, clearance_min, time_limit_s=60):
    n=len(df)
    conflict_sets=[set() for _ in range(n)]
    for i,j in conflicts:
        conflict_sets[i].add(j); conflict_sets[j].add(i)
    # Harder trains first: fewer eligible choices, then more conflicts.
    order=sorted(range(n), key=lambda i:(len(df.iloc[i].eligible_platforms), -len(conflict_sets[i])))
    assignment={}
    best_cost=float("inf")
    best_assignment=None
    nodes=0
    timed_out=False
    start=time.perf_counter()
    nominal=[int(df.iloc[i].nominal_platform) for i in range(n)]

    def dfs(k, cost):
        nonlocal best_cost,best_assignment,nodes,timed_out
        if time.perf_counter()-start > time_limit_s:
            timed_out=True; return
        nodes += 1
        if cost >= best_cost: return
        if k==n:
            best_cost=cost
            best_assignment=assignment.copy()
            return
        i=order[k]
        # Try nominal platform first to find a low-cost incumbent quickly.
        choices=list(df.iloc[i].eligible_platforms)
        choices.sort(key=lambda p:(p != nominal[i], p))
        for p in choices:
            if any(j in assignment and assignment[j]==p for j in conflict_sets[i]):
                continue
            new_cost=cost+(0 if p==nominal[i] else 1)
            if new_cost >= best_cost: continue
            assignment[i]=int(p)
            dfs(k+1,new_cost)
            del assignment[i]
            if timed_out: return

    dfs(0,0)
    elapsed=time.perf_counter()-start
    status = "TIMEOUT" if timed_out else ("OPTIMAL" if best_assignment is not None else "INFEASIBLE")
    validated = None
    if best_assignment is not None:
        validated, violations = validate_assignment(best_assignment, df, conflicts, clearance_min)
        if not validated:
            raise AssertionError(f"Exact solver returned invalid assignment: {violations}")
    return {
        "status": status,
        "best_cost": None if best_assignment is None else int(best_cost),
        "assignment": best_assignment,
        "solution_valid": validated,
        "runtime_s": elapsed,
        "nodes": nodes,
    }

conf_preview = find_conflicts(preview, BASE_CLEARANCE_MIN)
exact_preview = exact_backtracking(preview, conf_preview, BASE_CLEARANCE_MIN, time_limit_s=10)
exact_preview

{'status': 'OPTIMAL',
 'best_cost': 0,
 'assignment': {1: 2, 6: 2, 2: 5, 0: 0, 3: 3, 5: 4, 4: 1, 7: 0},
 'solution_valid': True,
 'runtime_s': 0.0006717000069329515,
 'nodes': 9}

## 6. SA and SQA solver wrappers

Both methods receive the same QUBO. Runtime includes the sampler call but excludes scenario generation and QUBO construction. OpenJij sampling is CPU-based in this notebook.

In [12]:
def decode_sample(sample, variable_labels):
    # OpenJij sample keys are the tuple labels used in QUBO.
    return {tuple(k): int(v) for k,v in sample.items()}


def evaluate_binary_sample(sample, df, conflicts, clearance_min):
    assignment={}
    for i in range(len(df)):
        selected=[p for p in df.iloc[i].eligible_platforms if int(sample.get(f'x_{i}_{int(p)}',0))==1]
        if len(selected)!=1:
            return False, None, None
        assignment[i]=int(selected[0])
    valid, violations=validate_assignment(assignment,df,conflicts,clearance_min)
    if not valid: return False, None, violations
    cost=sum(assignment[i] != int(df.iloc[i].nominal_platform) for i in range(len(df)))
    return True, int(cost), assignment


def run_openjij_solver(df, conflicts, clearance_min, penalty, solver="SA", num_reads=100, num_sweeps=500, seed=123):
    Q, labels, offset = build_qubo(df, conflicts, penalty)
    solver = solver.upper()
    if solver not in {"SA", "SQA"}:
        raise ValueError("solver must be SA or SQA")
    sampler = oj.SASampler() if solver == "SA" else oj.SQASampler()
    kwargs = {"num_reads": int(num_reads), "num_sweeps": int(num_sweeps), "seed": int(seed)}
    start = time.perf_counter()
    response = sampler.sample_qubo(Q, **kwargs)
    runtime = time.perf_counter() - start

    valid_occ = 0
    total_occ = 0
    costs = []
    best_assignment = None
    best_cost = float("inf")
    var_names = list(response.variables)
    for sample_arr, energy, occ in response.record:
        sample = dict(zip(var_names, sample_arr))
        total_occ += int(occ)
        ok, cost, assignment = evaluate_binary_sample(sample, df, conflicts, clearance_min)
        if ok:
            valid_occ += int(occ)
            costs.extend([cost] * int(occ))
            if cost < best_cost:
                best_cost = cost
                best_assignment = assignment
    return {
        "solver": solver,
        "runtime_s": runtime,
        "feasibility_pct": 100 * valid_occ / total_occ if total_occ else 0.0,
        "mean_valid_cost": float(np.mean(costs)) if costs else np.nan,
        "best_valid_cost": float(best_cost) if costs else np.nan,
        "total_reads": total_occ,
        "valid_reads": valid_occ,
        "best_assignment": best_assignment,
    }

# Smoke test on preview instance. Increase reads/sweeps for final experiments.
if exact_preview["status"] == "OPTIMAL":
    sa_preview = run_openjij_solver(preview, conf_preview, BASE_CLEARANCE_MIN, PENALTY_LAMBDA, "SA", NUM_READS, NUM_SWEEPS, 101)
    print("SA:", {k:v for k,v in sa_preview.items() if k!="best_assignment"})
    if RUN_SQA:
        sqa_preview = run_openjij_solver(preview, conf_preview, BASE_CLEARANCE_MIN, PENALTY_LAMBDA, "SQA", NUM_READS, NUM_SWEEPS, 102)
        print("SQA:", {k:v for k,v in sqa_preview.items() if k!="best_assignment"})

SA: {'solver': 'SA', 'runtime_s': 0.05120510001142975, 'feasibility_pct': 100.0, 'mean_valid_cost': 6.0, 'best_valid_cost': 6.0, 'total_reads': 100, 'valid_reads': 100}
SQA: {'solver': 'SQA', 'runtime_s': 0.675582800002303, 'feasibility_pct': 100.0, 'mean_valid_cost': 6.0, 'best_valid_cost': 6.0, 'total_reads': 100, 'valid_reads': 100}


## 7. Small-instance three-way comparison

Run Exact Search, SA and SQA on identical instances. The pilot uses fewer seeds/reads than the planned final paper; increase them only after the code passes validation.

In [13]:
def run_three_way_instance(n_trains, seed, repeat_id=0, long_platform_count=4, long_train_count=None,
                           delay_max_min=10, clearance_min=BASE_CLEARANCE_MIN,
                           penalty=PENALTY_LAMBDA, exact_limit_s=EXACT_TIME_LIMIT_S,
                           num_reads=NUM_READS, num_sweeps=NUM_SWEEPS, run_sqa=True):
    """One dataset seed and one independent SA/SQA repeat on the identical instance."""
    df = generate_jaipur_profile_instance(n_trains, seed, long_platform_count, long_train_count, delay_max_min, clearance_min)
    conflicts = find_conflicts(df, clearance_min)
    exact = exact_backtracking(df, conflicts, clearance_min, exact_limit_s)
    sa = run_openjij_solver(df, conflicts, clearance_min, penalty, "SA", num_reads, num_sweeps, seed + 10000 + repeat_id)
    rows = []
    rows.append({"n_trains": n_trains, "dataset_seed": seed, "repeat": repeat_id, "algorithm": "Exact",
                 "status": exact["status"], "solution_valid": exact["solution_valid"],
                 "best_cost": exact["best_cost"], "mean_valid_cost": np.nan,
                 "feasibility_pct": np.nan, "runtime_s": exact["runtime_s"], "nodes": exact["nodes"],
                 "optimality_gap": 0.0 if exact["status"] == "OPTIMAL" else np.nan,
                 "conflict_pairs": len(conflicts), "variables": sum(len(v) for v in df.eligible_platforms)})
    for alg, result in [("SA", sa)]:
        rows.append({"n_trains": n_trains, "dataset_seed": seed, "repeat": repeat_id, "algorithm": alg,
                     "status": "SAMPLED", "solution_valid": None, "best_cost": result["best_valid_cost"],
                     "mean_valid_cost": result["mean_valid_cost"], "feasibility_pct": result["feasibility_pct"],
                     "runtime_s": result["runtime_s"], "nodes": np.nan,
                     "optimality_gap": (result["best_valid_cost"] - exact["best_cost"]) if exact["status"] == "OPTIMAL" and np.isfinite(result["best_valid_cost"]) else np.nan,
                     "conflict_pairs": len(conflicts), "variables": sum(len(v) for v in df.eligible_platforms)})
    if run_sqa:
        sqa = run_openjij_solver(df, conflicts, clearance_min, penalty, "SQA", num_reads, num_sweeps, seed + 20000 + repeat_id)
        rows.append({"n_trains": n_trains, "dataset_seed": seed, "repeat": repeat_id, "algorithm": "SQA",
                     "status": "SAMPLED", "solution_valid": None, "best_cost": sqa["best_valid_cost"],
                     "mean_valid_cost": sqa["mean_valid_cost"], "feasibility_pct": sqa["feasibility_pct"],
                     "runtime_s": sqa["runtime_s"], "nodes": np.nan,
                     "optimality_gap": (sqa["best_valid_cost"] - exact["best_cost"]) if exact["status"] == "OPTIMAL" and np.isfinite(sqa["best_valid_cost"]) else np.nan,
                     "conflict_pairs": len(conflicts), "variables": sum(len(v) for v in df.eligible_platforms)})
    return pd.DataFrame(rows), df, conflicts


# Repeated pilot: same dataset seed is reused across repeats; solver seeds differ.
pilot_rows = []
if RUN_FULL_PILOT:
    for n_trains in [4, 6, 8, 10]:
        for dataset_seed in DATASET_SEEDS:
            for repeat_id in range(SOLVER_REPEATS):
                print(f"Running n={n_trains}, dataset_seed={dataset_seed}, repeat={repeat_id+1}/{SOLVER_REPEATS}")
                res, _, _ = run_three_way_instance(n_trains, dataset_seed, repeat_id=repeat_id,
                    exact_limit_s=EXACT_TIME_LIMIT_S, num_reads=NUM_READS,
                    num_sweeps=NUM_SWEEPS, run_sqa=RUN_SQA)
                pilot_rows.append(res)
pilot_results = pd.concat(pilot_rows, ignore_index=True) if pilot_rows else pd.DataFrame()

# Summarize repeated heuristic runs; exact appears once per repeated run for transparent timing.
if not pilot_results.empty:
    pilot_summary = (pilot_results.groupby(["n_trains", "algorithm"], dropna=False)
        .agg(runs=("repeat", "count"), median_best_cost=("best_cost", "median"),
             mean_best_cost=("best_cost", "mean"), sd_best_cost=("best_cost", "std"),
             median_runtime_s=("runtime_s", "median"), median_feasibility_pct=("feasibility_pct", "median"),
             median_optimality_gap=("optimality_gap", "median"))
        .reset_index())
else:
    pilot_summary = pd.DataFrame()

pilot_summary


Running n=4, dataset_seed=11, repeat=1/3
Running n=4, dataset_seed=11, repeat=2/3
Running n=4, dataset_seed=11, repeat=3/3
Running n=4, dataset_seed=22, repeat=1/3
Running n=4, dataset_seed=22, repeat=2/3
Running n=4, dataset_seed=22, repeat=3/3
Running n=4, dataset_seed=33, repeat=1/3
Running n=4, dataset_seed=33, repeat=2/3
Running n=4, dataset_seed=33, repeat=3/3
Running n=6, dataset_seed=11, repeat=1/3
Running n=6, dataset_seed=11, repeat=2/3
Running n=6, dataset_seed=11, repeat=3/3
Running n=6, dataset_seed=22, repeat=1/3
Running n=6, dataset_seed=22, repeat=2/3
Running n=6, dataset_seed=22, repeat=3/3
Running n=6, dataset_seed=33, repeat=1/3
Running n=6, dataset_seed=33, repeat=2/3
Running n=6, dataset_seed=33, repeat=3/3
Running n=8, dataset_seed=11, repeat=1/3
Running n=8, dataset_seed=11, repeat=2/3
Running n=8, dataset_seed=11, repeat=3/3
Running n=8, dataset_seed=22, repeat=1/3
Running n=8, dataset_seed=22, repeat=2/3
Running n=8, dataset_seed=22, repeat=3/3
Running n=8, dat

,n_trains,algorithm,runs,median_best_cost,mean_best_cost,sd_best_cost,median_runtime_s,median_feasibility_pct,median_optimality_gap
0,4,Exact,9,0.0,0.000000,0.000000,0.000444,NaN,0.0
1,4,SA,9,4.0,3.444444,0.881917,0.038226,100.0,4.0
2,4,SQA,9,2.0,2.111111,1.269296,0.679725,100.0,2.0
3,6,Exact,9,0.0,0.000000,0.000000,0.000505,NaN,0.0
4,6,SA,9,5.0,4.555556,1.424001,0.037214,100.0,5.0
5,6,SQA,9,4.0,4.444444,0.881917,0.711907,100.0,4.0
6,8,Exact,9,0.0,0.000000,0.000000,0.000471,NaN,0.0
7,8,SA,9,7.0,6.333333,1.802776,0.037539,100.0,7.0
8,8,SQA,9,5.0,5.000000,1.581139,0.547098,100.0,5.0
9,10,Exact,9,0.0,0.000000,0.000000,0.000597,NaN,0.0


## 8. Scaling beyond the exact-search range

For larger instances, run SA and SQA on identical generated scenarios. Exact Search may be attempted with a time limit, but timeouts must be recorded as timeouts, not as optimal solutions. The default list is a conservative pilot; increase gradually after measuring runtime and memory.

In [14]:
def run_heuristic_scaling(n_values, seeds, long_platform_count=4, long_train_count=None,
                          delay_max_min=10, clearance_min=BASE_CLEARANCE_MIN,
                          penalty=PENALTY_LAMBDA, num_reads=NUM_READS,
                          num_sweeps=NUM_SWEEPS, run_sqa=True):
    rows=[]
    for n in n_values:
        for seed in seeds:
            df=generate_jaipur_profile_instance(n,seed,long_platform_count,long_train_count,delay_max_min,clearance_min)
            conflicts=find_conflicts(df,clearance_min)
            for alg, offset in [("SA",30000),("SQA",40000)]:
                if alg=="SQA" and not run_sqa: continue
                r=run_openjij_solver(df,conflicts,clearance_min,penalty,alg,num_reads,num_sweeps,seed+offset)
                rows.append({"n_trains":n,"n_variables":sum(len(v) for v in df.eligible_platforms),
                             "seed":seed,"algorithm":alg,"best_cost":r["best_valid_cost"],
                             "mean_valid_cost":r["mean_valid_cost"],"feasibility_pct":r["feasibility_pct"],
                             "runtime_s":r["runtime_s"],"conflict_pairs":len(conflicts)})
            print(f"Completed n={n}, seed={seed}")
    return pd.DataFrame(rows)

# Run after the small pilot succeeds. Do not start with every stress size on the first run.
RUN_LARGE_SCALING = False
large_results=pd.DataFrame()
if RUN_LARGE_SCALING:
    large_results=run_heuristic_scaling([12,16,20,24,28,32,36,40], seeds=[11,22,33], run_sqa=RUN_SQA)
large_results.head()

""


## 9. Penalty, delay and platform-capacity sensitivity

Use the same generated instances across settings. This prevents differences in random train schedules from being mistaken for algorithm effects.

In [15]:
def run_sensitivity(kind="penalty", n_trains=8, seed=77, values=None, long_platform_count=4):
    if values is None:
        values = [5,10,15,20,30,40] if kind=="penalty" else ([0,5,10,15] if kind=="delay" else [4,5])
    rows=[]
    for val in values:
        if kind=="penalty":
            delay=10; lpc=long_platform_count; penalty=float(val)
        elif kind=="delay":
            delay=int(val); lpc=long_platform_count; penalty=PENALTY_LAMBDA
        elif kind=="capacity":
            delay=10; lpc=int(val); penalty=PENALTY_LAMBDA
        else: raise ValueError("kind must be penalty, delay or capacity")
        df=generate_jaipur_profile_instance(n_trains,seed,lpc,delay_max_min=delay)
        conflicts=find_conflicts(df,BASE_CLEARANCE_MIN)
        for alg,offset in [("SA",50000),("SQA",60000)]:
            if alg=="SQA" and not RUN_SQA: continue
            r=run_openjij_solver(df,conflicts,BASE_CLEARANCE_MIN,penalty,alg,NUM_READS,NUM_SWEEPS,seed+offset)
            rows.append({"kind":kind,"setting":val,"algorithm":alg,"feasibility_pct":r["feasibility_pct"],
                         "best_cost":r["best_valid_cost"],"mean_valid_cost":r["mean_valid_cost"],"runtime_s":r["runtime_s"],
                         "conflict_pairs":len(conflicts),"variables":sum(len(v) for v in df.eligible_platforms)})
    return pd.DataFrame(rows)

# Penalty sensitivity uses one fixed generated instance across all lambda values.
# Interpret feasibility and valid objective quality separately; a high feasibility rate
# does not by itself demonstrate that the penalty preserves the desired optimum.

# Examples (uncomment one at a time):
# penalty_sensitivity = run_sensitivity("penalty", n_trains=8)
# delay_sensitivity = run_sensitivity("delay", n_trains=12)
# capacity_sensitivity = run_sensitivity("capacity", n_trains=12)


## 10. Visualize and export results

In [16]:
def plot_scaling(results):
    if results.empty: print("No scaling results to plot"); return
    fig,ax=plt.subplots(figsize=(8,5))
    for alg,g in results.groupby("algorithm"):
        stats=g.groupby("n_trains").runtime_s.median()
        ax.plot(stats.index,stats.values,marker="o",label=alg)
    ax.set_xlabel("Number of trains"); ax.set_ylabel("Median sampler runtime (seconds)")
    ax.set_title("SA vs SQA runtime scaling"); ax.grid(True,alpha=.3); ax.legend(); plt.show()


def export_results(*frames, prefix="jaipur_benchmark"):
    nonempty=[f for f in frames if f is not None and not f.empty]
    if not nonempty: print("No result frames to export"); return
    out=pd.concat(nonempty,ignore_index=True,sort=False)
    out.to_csv(f"{prefix}_results.csv",index=False)
    out.to_json(f"{prefix}_results.json",orient="records",indent=2)
    print("Exported CSV and JSON; rows:",len(out))
    return out

if not pilot_results.empty:
    display(pilot_results)
# plot_scaling(large_results)
# all_results = export_results(pilot_results, large_results)


,n_trains,dataset_seed,repeat,algorithm,status,solution_valid,best_cost,mean_valid_cost,feasibility_pct,runtime_s,nodes,optimality_gap,conflict_pairs,variables
0,4,11,0,Exact,OPTIMAL,True,0.0,NaN,NaN,0.000415,5.0,0.0,6,22
1,4,11,0,SA,SAMPLED,None,4.0,4.0,100.0,0.029120,NaN,4.0,6,22
2,4,11,0,SQA,SAMPLED,None,3.0,3.0,100.0,0.597690,NaN,3.0,6,22
3,4,11,1,Exact,OPTIMAL,True,0.0,NaN,NaN,0.000816,5.0,0.0,6,22
4,4,11,1,SA,SAMPLED,None,4.0,4.0,100.0,0.038226,NaN,4.0,6,22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103,10,33,1,SA,SAMPLED,None,NaN,NaN,0.0,0.048051,NaN,NaN,35,56
104,10,33,1,SQA,SAMPLED,None,NaN,NaN,0.0,0.567697,NaN,NaN,35,56
105,10,33,2,Exact,INFEASIBLE,None,NaN,NaN,NaN,0.039322,785.0,NaN,35,56
106,10,33,2,SA,SAMPLED,None,NaN,NaN,0.0,0.064949,NaN,NaN,35,56


## 11. Reproducibility and reporting checklist

Before using results in the paper, record: (a) notebook version/commit; (b) Python, NumPy, OpenJij versions; (c) CPU model, RAM and OS; (d) whether CUDA was used and for which exact-search operation; (e) seeds; (f) number of reads and sweeps; (g) penalty values; (h) train counts and eligible-variable counts; (i) number of conflict pairs; (j) exact-search status, explored nodes and timeout; (k) solver runtimes and validation results.

**Do not report the pilot settings as final methodology.** Increase from 3 seeds/100 reads/500 sweeps to the planned 10 seeds/300 reads/1000 sweeps only after the pilot is stable. Keep the same settings for SA and SQA. Report exact-search timeouts honestly. Do not claim a physical quantum computer or quantum advantage.

## 12. Interpretation safeguards added in v2

- A random-bitstring identity test compares the QUBO energy (including its offset) with an independently evaluated objective-plus-penalties expression. This checks coefficient construction, not whether the chosen penalty is large enough to preserve the constrained optimum.
- The pilot now uses every configured dataset seed and independent solver repeat. Each repeat uses the same dataset for Exact, SA and SQA, while sampler seeds differ.
- Exact feasibility is represented by `solution_valid` and solver status; its `feasibility_pct` is left undefined because Exact does not produce a read distribution.
- `optimality_gap` is reported only when Exact completed with `OPTIMAL` status. For this minimization objective, it is heuristic best cost minus exact best cost.
- Penalty calibration still requires empirical sensitivity analysis and, ideally, comparison with exact optima on multiple small instances. Passing the QUBO identity test is necessary but not sufficient.
- The synthetic generator's disruption severity must be characterized using observed conflict counts and required reassignment costs; do not tune it merely to force nonzero costs.
